In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [2]:

from langgraph.checkpoint.memory import InMemorySaver

# 메모리 종류 

# short term memory: Checkpointer를 통해 대화 스레드(세션) 저장
# long term memory: Store를 통해 장기 메모리 저장 

# 주의할 점은 invoke할 때 thread_id = 1를 명시적으로 지정해야 한다는 점입니다. 

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver()
    
    )


In [3]:
result = agent.invoke({"messages": [
  {"role": "user", "content": "안녕하세요, 나는 자유인입니다"}]},
  {"configurable": {"thread_id": 1}})

print(result['messages'][-1].content)

안녕하세요! 자유인님. 어떻게 도와드릴까요?


In [4]:
result = agent.invoke({"messages": [
  {"role": "user", "content": "제가 뭐라고 했죠 ?"}]},
  {"configurable": {"thread_id": 1}}) # thread_id가 다르면, 새로운 대화로 인식합니다.


print(result['messages'][-1].content)



당신은 "안녕하세요, 나는 자유인입니다"라고 말씀하셨습니다. 자유로운 삶에 대해 이야기하고 싶으신가요? 아니면 다른 주제가 있으신가요?


In [5]:
result = agent.invoke({"messages": [
  {"role": "user", "content": "제가 뭐라고 했죠 ?"}]},
  {"configurable": {"thread_id": 2}}) # thread_id가 다르면, 새로운 대화로 인식합니다.


print(result['messages'][-1].content)



죄송하지만, 저는 이전 대화 내용을 기억할 수 없습니다. 제가 도와드릴 수 있는 것이 있다면 말씀해 주세요!


In [6]:
# 1번 방의 대화 내역을 다시 불러옵니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "지금까지 무슨 얘기 나눴죠?"}]},
    {"configurable": {"thread_id": "1"}},
)

# 세션에 누적된 전체 메시지 히스토리를 하나씩 뜯어봅니다.
for i, msg in enumerate(response["messages"], start=1):
    print(f"--- Message {i} ({msg.type}) ---")
    print(msg.content)
    print()


--- Message 1 (human) ---
안녕하세요, 나는 자유인입니다

--- Message 2 (ai) ---
안녕하세요! 자유인님. 어떻게 도와드릴까요?

--- Message 3 (human) ---
제가 뭐라고 했죠 ?

--- Message 4 (ai) ---
당신은 "안녕하세요, 나는 자유인입니다"라고 말씀하셨습니다. 자유로운 삶에 대해 이야기하고 싶으신가요? 아니면 다른 주제가 있으신가요?

--- Message 5 (human) ---
지금까지 무슨 얘기 나눴죠?

--- Message 6 (ai) ---
지금까지 당신은 "안녕하세요, 나는 자유인입니다"라고 인사하셨고, 제가 그에 대해 반응하며 어떻게 도와드릴지 물었습니다. 그 이후에 당신이 제가 지금까지의 대화를 요약해 달라고 요청하셨습니다. 더 이야기하고 싶은 주제가 있으면 말씀해 주세요!

